From one source

In [1]:
import json 
import sys
import os

# Add workspace root to path for absolute imports
workspace_root = os.path.abspath("../..")
sys.path.insert(0, workspace_root)

DATA_ROOT = "../test_sets/"


from evaluation.metrics.evaluator import (
    GenerationEvaluator, 
    RetrieverEvaluator,
    EvaluatorConfig,
    EvaluationResult
)
from evaluation.metrics.trad_metrics import (
    unrankedMetrics,
    mean_reciprocal_rank,
    ndcg_at_k,
    mean_average_precision,
    exact_match,
    meteor_batch
)

ModuleNotFoundError: No module named 'trad_metrics'

In [ ]:
with open(DATA_ROOT + 'final_dataset.json') as f:
    dataset = json.load(f)


print('format :', list(dataset[0].keys()))

format : ['Question', 'Chunks', 'Answer']


In [ ]:
# Import evaluators and metrics from evaluator.py
import sys
sys.path.insert(0, '.')



# Extract all answers and questions from dataset
all_questions = [sample['Question'] for sample in dataset]
all_answers = [sample['Answer'] for sample in dataset]
all_chunks = [sample['Chunks'] for sample in dataset]

print(f"Dataset: {len(dataset)} samples")
print(f"Extracted: {len(all_answers)} answers, {len(all_questions)} questions")

Dataset: 212 samples
Extracted: 212 answers, 212 questions


In [ ]:
# Generation Evaluator - Traditional NLP metrics
print("\n1. GENERATION METRICS (Exact Match, METEOR)")

gen_evaluator = GenerationEvaluator()
predictions = all_answers.copy()
references = all_answers.copy()
gen_result = gen_evaluator.evaluate(predictions, references)

print(f"Perfect predictions: EM={gen_result.metrics.get('exact_match', 0):.3f}, METEOR={gen_result.metrics.get('meteor', 0):.3f}")


1. GENERATION METRICS (Exact Match, METEOR)
Perfect predictions: EM=1.000, METEOR=0.775


In [ ]:
# Retriever Evaluator - Ranking metrics
print("\n2. RETRIEVER METRICS (MRR, NDCG, MAP)")

predictions_list = [[0, 1, 2, 3, 4], [0, 2, 1, 3], [1, 0, 3, 2, 4]]
ground_truth_list = [[0], [0, 2], [1, 3]]

config = EvaluatorConfig(name="RetrieverEvaluator", additional_params={"k_values": [3, 5]})
retriever_evaluator = RetrieverEvaluator(config)
retriever_result = retriever_evaluator.evaluate(predictions_list=predictions_list, ground_truth_list=ground_truth_list, k_values=[3, 5])

metrics = retriever_result.metrics
if "rank_based" in metrics:
    rbm = metrics["rank_based"]
    print(f"Rank-based: MRR={rbm.get('mrr', 0):.3f}, NDCG@5={rbm.get('ndcg@5', 0):.3f}, MAP@5={rbm.get('map@5', 0):.3f}")
if "non_rank_based" in metrics:
    nbm = metrics["non_rank_based"]
    print(f"Non-rank-based: Accuracy={nbm.get('accuracy', 0):.3f}, Recall={nbm.get('recall', 0):.3f}, F1={nbm.get('f1', 0):.3f}")


2. RETRIEVER METRICS (MRR, NDCG, MAP)
Rank-based: MRR=1.000, NDCG@5=0.973, MAP@5=0.944
Non-rank-based: Accuracy=1.000, Recall=1.000, F1=0.333


In [ ]:
# Comparative Analysis
print("\n3. SCENARIO COMPARISON")

perfect_preds = references
result_perfect = gen_evaluator.evaluate(perfect_preds, references)
print(f"Perfect: EM={result_perfect.metrics.get('exact_match', 0):.3f}, METEOR={result_perfect.metrics.get('meteor', 0):.3f}")

partial_preds = [answer.replace("an ", "a ") if "an " in answer else answer for answer in references]
result_partial = gen_evaluator.evaluate(partial_preds, references)
print(f"Partial: EM={result_partial.metrics.get('exact_match', 0):.3f}, METEOR={result_partial.metrics.get('meteor', 0):.3f}")

import random
random.seed(42)
poor_preds = [references[i][:30] if len(references[i]) > 30 else references[i] for i in range(len(references))]
result_poor = gen_evaluator.evaluate(poor_preds, references)
print(f"Poor: EM={result_poor.metrics.get('exact_match', 0):.3f}, METEOR={result_poor.metrics.get('meteor', 0):.3f}")

print("\n4. SUMMARY")
print("GenerationEvaluator: Exact Match and METEOR for answer quality")
print("RetrieverEvaluator: Ranking metrics (MRR, NDCG, MAP) and retrieval metrics (Accuracy, Recall)")


3. SCENARIO COMPARISON
Perfect: EM=1.000, METEOR=0.775
Partial: EM=0.679, METEOR=0.759
Poor: EM=0.208, METEOR=0.243

4. SUMMARY
GenerationEvaluator: Exact Match and METEOR for answer quality
RetrieverEvaluator: Ranking metrics (MRR, NDCG, MAP) and retrieval metrics (Accuracy, Recall)


In [ ]:
# Test DatasetEvaluator
print("\n5. DATASET QUALITY METRICS")

from evaluator import DatasetEvaluator

dataset_evaluator = DatasetEvaluator()

# Prepare dataset in expected format: 'question', 'chunks', 'answer'
eval_dataset = [
    {
        'question': item['Question'],
        'chunks': item['Chunks'],
        'answer': item['Answer']
    }
    for item in dataset
]

dataset_result = dataset_evaluator.evaluate(eval_dataset)

print(f"Question diversity: {dataset_result.metrics.get('question_diversity', 0):.3f}")
print(f"Chunk diversity: {dataset_result.metrics.get('chunk_diversity', 0):.3f}")
print(f"Source diversity: {dataset_result.metrics.get('source_diversity', 0):.3f}")

align = dataset_result.metrics.get('answer_chunk_alignment', {})
print(f"Answer-chunk alignment: mean={align.get('mean_coverage', 0):.3f}, min={align.get('min_coverage', 0):.3f}, max={align.get('max_coverage', 0):.3f}")
print(f"Samples: {dataset_result.metadata.get('num_samples', 0)}, Chunks: {dataset_result.metadata.get('total_chunks', 0)}")


5. DATASET QUALITY METRICS


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\malopif\miniconda3\envs\AI\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\malopif\.cache\huggingface\hub\models--Mihaiii--Ivysaur. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

c:\Users\malopif\miniconda3\envs\AI\Lib\site-packages\torch\cuda\__init__.py:262: UserWarning: 
    Found GPU0 NVIDIA GeForce MX150 which is of cuda capability 6.1.
    PyTorch no longer supports this GPU because it is too old.
    The minimum cuda capability supported by this library is 7.5.
    
  warnings.warn(
c:\Users\malopif\miniconda3\envs\AI\Lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce MX150 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce MX150 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
DatasetEvaluator: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider pa

Question diversity: 0.000
Chunk diversity: 0.000
Source diversity: 0.000
Answer-chunk alignment: mean=0.000, min=0.000, max=0.000
Samples: 0, Chunks: 0
